# Home Credit Risk Analysis

## Dataset: POS_CASH_balance.csv

### Objetivo

Analizar la estructura, calidad y relaciones del dataset para comprender su papel dentro de la arquitectura Lakehouse y del modelo dimensional.

---

### Contexto del negocio

El dataset `POS_CASH_balance.csv` contiene el historial mensual de comportamiento asociado a préstamos de tipo Point of Sale (POS) y Cash Loans otorgados a los clientes de Home Credit.

Cada registro representa el estado de un préstamo en un período específico e incluye información relacionada con cuotas pendientes, estado contractual, niveles de atraso y evolución del crédito a lo largo del tiempo.

La información almacenada permite analizar el cumplimiento de obligaciones financieras, el avance de los planes de pago y el comportamiento histórico de los clientes frente a este tipo de productos crediticios.

A diferencia de las tarjetas de crédito, estos préstamos suelen estar asociados a planes de pago estructurados mediante cuotas definidas.

---

### Rol dentro de la Arquitectura

El dataset forma parte de la capa histórica de comportamiento financiero y complementa la información proveniente de solicitudes, créditos y pagos.

Su función principal es proporcionar visibilidad sobre la evolución mensual de préstamos POS y Cash Loans.

application_train
        │
        ▼
previous_application
        │
        ▼
POS_CASH_balance

La información contenida en esta fuente permitirá construir indicadores relacionados con:

- Cumplimiento de cuotas.
- Evolución de créditos activos.
- Niveles de atraso.
- Historial de mora.
- Comportamiento de pago.
- Riesgo asociado a préstamos POS y Cash Loans.

# 1. Importación de librerías

Se cargan las librerías necesarias para el análisis exploratorio de datos.

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)

# 2. Carga del Dataset

Se carga el dataset para realizar el análisis exploratorio y comprender su estructura, calidad y relevancia dentro del dominio de negocio.

In [2]:
df = pd.read_csv("../../../data/raw/homeCredit/POS_CASH_balance.csv")

print(f"Filas: {df.shape[0]}")
print(f"Columnas: {df.shape[1]}")

FileNotFoundError: [Errno 2] No such file or directory: '../../../data/raw/homeCredit/POS_CASH_balance.csv'

# 3. Tamaño del Dataset

## Resultados

| Métrica | Valor |
|----------|---------|
| Filas | 10,001,358 |
| Columnas | 8 |

## Interpretación

El dataset `POS_CASH_balance.csv` contiene 10,001,358 registros distribuidos en 8 columnas.

El volumen de información evidencia que se trata de una fuente histórica orientada al seguimiento periódico de préstamos de tipo Point of Sale (POS) y Cash Loans otorgados a los clientes de Home Credit.

Cada registro representa el estado de un préstamo en un período específico, permitiendo analizar la evolución temporal de cuotas pendientes, niveles de atraso y comportamiento de pago.

---

### Características Generales

El dataset almacena información relacionada con:

- Créditos POS y Cash Loans.
- Estado mensual de los préstamos.
- Cuotas pendientes.
- Cuotas futuras.
- Historial de atraso.
- Evolución contractual.

---

### Relevancia para el Negocio

La información contenida en este dataset permite comprender cómo evolucionan los préstamos otorgados a los clientes a lo largo del tiempo.

A diferencia de previous_application, que registra la solicitud inicial del crédito, POS_CASH_balance permite monitorear el comportamiento posterior del préstamo durante su ciclo de vida.

---

### Relevancia para el Riesgo Crediticio

El comportamiento observado en los préstamos constituye una fuente importante para evaluar el riesgo financiero de los clientes.

A partir de esta información será posible construir indicadores relacionados con:

- Cumplimiento de cuotas.
- Historial de atraso.
- Evolución de préstamos activos.
- Frecuencia de mora.
- Comportamiento de pago.

---

### Consideraciones para Ingeniería de Datos

Con más de 10 millones de registros, POS_CASH_balance constituye una fuente transaccional de gran volumen dentro del ecosistema Home Credit.

Su naturaleza histórica y periódica la convierte en una candidata ideal para procesos de agregación y generación de indicadores analíticos dentro de las capas Silver, Intermediate y Gold de la arquitectura Lakehouse.

# 4. Vista Preliminar

Se inspeccionan los primeros registros del dataset con el objetivo de comprender su estructura, contenido y nivel de granularidad.

## Observaciones Iniciales

La muestra evidencia que cada registro representa el estado mensual de un préstamo POS o Cash Loan asociado a un cliente.

El dataset almacena información relacionada con el avance del plan de cuotas, estado contractual y niveles de atraso observados durante la vida del crédito.

Su naturaleza periódica permite analizar la evolución histórica de los préstamos y el comportamiento de cumplimiento de los clientes.

---

## Variables Observadas

| Variable | Descripción |
|-----------|-------------|
| SK_ID_PREV | Identificador del crédito asociado |
| SK_ID_CURR | Identificador del cliente |
| MONTHS_BALANCE | Mes relativo de observación |
| CNT_INSTALMENT | Número total de cuotas del crédito |
| CNT_INSTALMENT_FUTURE | Número de cuotas pendientes |
| NAME_CONTRACT_STATUS | Estado del contrato |
| SK_DPD | Días de atraso |
| SK_DPD_DEF | Días de atraso considerados como mora |

---

## Hallazgos Iniciales

La estructura observada permite analizar simultáneamente el avance del crédito y el comportamiento de pago del cliente.

---

### Evolución del Plan de Cuotas

Las variables:

- CNT_INSTALMENT
- CNT_INSTALMENT_FUTURE

permiten medir el progreso del crédito a lo largo del tiempo.

Por ejemplo:

| Cuotas Totales | Cuotas Pendientes |
|---------------|------------------|
| 48 | 45 |
| 36 | 35 |
| 12 | 9 |

Estos valores sugieren que el dataset registra diferentes momentos dentro del ciclo de vida de cada préstamo.

---

### Estado Contractual

La variable:

- NAME_CONTRACT_STATUS

describe la situación operativa del crédito durante cada período observado.

En la muestra inicial todos los registros presentan estado **Active**, indicando créditos que permanecían vigentes en el momento de la observación.

---

### Niveles de Atraso

Las variables:

- SK_DPD
- SK_DPD_DEF

permiten evaluar situaciones de incumplimiento financiero.

En la muestra observada no se registran días de atraso, lo que indica cumplimiento normal de las obligaciones asociadas al crédito.

---

## Primera Hipótesis de Granularidad

A partir de los registros observados se plantea la siguiente hipótesis:

1 registro = 1 crédito en 1 período mensual

Esta hipótesis será validada posteriormente mediante el análisis de claves y cardinalidad.

---

## Relación con Otros Datasets

Mientras que previous_application registra las solicitudes históricas e installments_payments registra eventos de pago específicos, POS_CASH_balance almacena el seguimiento mensual del estado general de los préstamos.

Esta característica convierte al dataset en una fuente estratégica para analizar la evolución de créditos activos y el comportamiento histórico de cumplimiento financiero.

---

## Potenciales Indicadores

La estructura observada permitirá construir métricas relacionadas con:

- Porcentaje de avance del crédito.
- Cuotas pendientes promedio.
- Historial de mora.
- Frecuencia de atraso.
- Créditos activos.
- Créditos completados.
- Riesgo asociado a préstamos POS y Cash Loans.

In [3]:
df.head()

NameError: name 'df' is not defined

# 5. Estructura General

Se analiza la composición estructural del dataset para comprender los tipos de datos utilizados, evaluar la calidad general del esquema y estimar la complejidad de los procesos analíticos posteriores.

---

## Resumen Estructural

| Métrica | Valor |
|----------|---------|
| Registros | 10,001,358 |
| Columnas | 8 |
| Variables Numéricas | 7 |
| Variables Categóricas | 1 |
| Tipos de Datos Distintos | 3 |

---

## Distribución de Tipos de Datos

| Tipo de Dato | Cantidad |
|--------------|----------|
| int64 | 5 |
| float64 | 2 |
| string | 1 |

---

## Interpretación

El dataset presenta una estructura compacta y altamente especializada para el seguimiento mensual de préstamos POS y Cash Loans.

La mayor parte de las variables corresponden a identificadores, métricas relacionadas con cuotas y variables de control de atraso.

Esta composición resulta consistente con la naturaleza operativa del dataset, cuyo objetivo es registrar la evolución mensual de créditos previamente otorgados.

---

## Componentes Principales

### Variables Identificadoras

Las columnas:

- SK_ID_PREV
- SK_ID_CURR

permiten relacionar cada observación con el crédito correspondiente y con el cliente asociado.

---

### Variable Temporal

La columna:

- MONTHS_BALANCE

permite analizar la evolución histórica de los créditos a lo largo del tiempo.

Cada registro representa una fotografía mensual del estado del préstamo.

---

### Variables de Cuotas

Las columnas:

- CNT_INSTALMENT
- CNT_INSTALMENT_FUTURE

permiten medir el avance del plan de pagos y determinar cuántas cuotas permanecen pendientes.

Estas variables proporcionan información sobre el ciclo de vida del crédito.

---

### Variables de Riesgo

Las columnas:

- SK_DPD
- SK_DPD_DEF

permiten identificar situaciones de atraso y mora.

Estas variables representan señales directas de riesgo crediticio y cumplimiento financiero.

---

### Variable Categórica

El dataset incorpora una única variable categórica:

- NAME_CONTRACT_STATUS

Esta columna describe el estado operativo del crédito durante cada período observado.

---

## Calidad Estructural

La estructura observada presenta una distribución equilibrada entre variables operativas, temporales y de riesgo.

La simplicidad del esquema facilita los procesos de transformación, agregación y construcción de indicadores históricos.

---

## Consideraciones para Ingeniería de Datos

Con más de 10 millones de registros y una estructura compacta de únicamente 8 columnas, POS_CASH_balance constituye una fuente eficiente para el procesamiento distribuido mediante Spark.

Su naturaleza periódica permitirá generar métricas históricas relacionadas con cumplimiento de cuotas, evolución de créditos y comportamiento de mora dentro de las capas Silver, Intermediate y Gold de la arquitectura Lakehouse.

In [4]:
df.info()

NameError: name 'df' is not defined

In [5]:
df.dtypes.value_counts()

NameError: name 'df' is not defined

# 6. Identificación de Claves

Se identifican:

- Clave primaria
- Claves foráneas
- Relaciones con otros datasets

In [6]:
len(df)

NameError: name 'df' is not defined

In [7]:
df["SK_ID_PREV"].nunique()

NameError: name 'df' is not defined

In [8]:
df["SK_ID_CURR"].nunique()

NameError: name 'df' is not defined

In [9]:
df.duplicated().sum()

NameError: name 'df' is not defined

In [10]:
df[
    ["SK_ID_PREV", "MONTHS_BALANCE"]
].duplicated().sum()

NameError: name 'df' is not defined

## Interpretación

El análisis de cardinalidad evidencia que ninguna de las columnas identificadoras principales puede actuar individualmente como clave primaria.

---

### SK_ID_PREV

La columna SK_ID_PREV presenta 936,325 valores únicos para un total de 10,001,358 registros.

Este comportamiento indica que un mismo crédito genera múltiples observaciones históricas a lo largo del tiempo.

Por lo tanto, SK_ID_PREV no constituye una clave primaria.

---

### SK_ID_CURR

La columna SK_ID_CURR presenta 337,252 valores únicos para un total de 10,001,358 registros.

Esto confirma que un mismo cliente puede registrar múltiples créditos y múltiples observaciones históricas.

Por lo tanto, SK_ID_CURR tampoco constituye una clave primaria.

---

### Integridad de Registros

No se identificaron registros completamente duplicados dentro del dataset.

| Métrica | Valor |
|----------|---------|
| Registros Totales | 10,001,358 |
| Registros Duplicados | 0 |

---

### Clave Primaria Candidata

El análisis permitió identificar una combinación de columnas que garantiza unicidad a nivel de registro:

| Columna |
|----------|
| SK_ID_PREV |
| MONTHS_BALANCE |

Validación realizada:

| Métrica | Valor |
|----------|---------|
| Duplicados (SK_ID_PREV + MONTHS_BALANCE) | 0 |

---

### Interpretación

La combinación entre SK_ID_PREV y MONTHS_BALANCE identifica de forma única cada observación dentro del dataset.

Esto indica que cada registro representa una fotografía mensual del estado de un préstamo POS o Cash Loan.

Por lo tanto, la granularidad observada puede definirse como:

1 registro = 1 crédito en 1 período mensual

---

### Claves Foráneas

#### SK_ID_CURR

Permite relacionar cada observación con el cliente correspondiente.

#### SK_ID_PREV

Permite relacionar cada observación con el crédito asociado.

---

### Cardinalidad

application_train (1)

↓

POS_CASH_balance (N)

Un cliente puede registrar múltiples créditos y múltiples observaciones mensuales asociadas a dichos créditos.

---

### Conclusión

La combinación SK_ID_PREV + MONTHS_BALANCE constituye una clave de negocio válida para identificar de forma única cada observación dentro del dataset.

Esta estructura confirma que POS_CASH_balance funciona como una tabla histórica de seguimiento mensual de créditos.

# 7. Calidad de Datos

Se analiza la completitud de la información y posibles problemas de calidad.

In [11]:
null_percentage = (
    df.isnull()
      .mean()
      .mul(100)
      .sort_values(ascending=False)
)

null_percentage.head(20)

NameError: name 'df' is not defined

In [12]:
high_nulls = null_percentage[null_percentage > 50]

medium_nulls = null_percentage[
    (null_percentage > 20)
    & (null_percentage <= 50)
]

low_nulls = null_percentage[
    (null_percentage > 0)
    & (null_percentage <= 20)
]

print("Columnas > 50%:", len(high_nulls))
print("Columnas entre 20%-50%:", len(medium_nulls))
print("Columnas < 20%:", len(low_nulls))

NameError: name 'null_percentage' is not defined

## Interpretación

El análisis de calidad de datos evidencia un nivel de completitud excepcional dentro del dataset.

De las ocho variables analizadas, seis presentan un 100% de completitud y únicamente dos columnas registran valores faltantes.

---

### Valores Nulos Identificados

| Columna | % Nulos |
|----------|---------|
| CNT_INSTALMENT_FUTURE | 0.260835% |
| CNT_INSTALMENT | 0.260675% |

---

### Distribución de Completitud

| Categoría | Cantidad |
|------------|----------|
| Columnas > 50% nulos | 0 |
| Columnas entre 20% y 50% | 0 |
| Columnas < 20% | 2 |

---

### Interpretación de Negocio

Los valores faltantes se concentran exclusivamente en variables relacionadas con el plan de cuotas del crédito.

Las columnas:

- CNT_INSTALMENT
- CNT_INSTALMENT_FUTURE

almacenan información sobre el número total de cuotas y las cuotas pendientes del préstamo.

La similitud observada en los porcentajes de valores faltantes sugiere una relación directa entre ambas variables.

---

### Hipótesis de Negocio

Los registros con valores faltantes podrían corresponder a:

- Créditos cerrados.
- Créditos cancelados.
- Productos financieros especiales.
- Situaciones operativas donde la información de cuotas no resulta aplicable.

Por lo tanto, estos valores nulos no necesariamente representan errores de calidad de datos.

---

### Hallazgos

- Más del 99.7% de los registros presentan información completa.
- No se identifican problemas significativos de completitud.
- Los valores faltantes se concentran únicamente en variables relacionadas con cuotas.
- La calidad observada facilita la construcción de indicadores históricos de cumplimiento financiero.

---

### Implicaciones para la Arquitectura Lakehouse

#### Bronze

Los registros deben conservarse sin modificaciones para mantener la trazabilidad de la información original.

#### Silver

Será necesario validar la naturaleza de los registros con valores faltantes e identificar si corresponden a situaciones operativas específicas.

#### Gold

Las variables relacionadas con cuotas permitirán construir métricas asociadas al avance y cumplimiento de los créditos.

---

### Conclusión

POS_CASH_balance presenta uno de los niveles de calidad más altos observados dentro del proyecto.

La baja proporción de valores faltantes y la consistencia estructural de sus variables lo convierten en una fuente altamente confiable para el análisis histórico de préstamos POS y Cash Loans.

# 8. Detección de Registros Duplicados

In [13]:
df.duplicated().sum()

NameError: name 'df' is not defined

## Interpretación

Se realizó una validación de duplicidad sobre la totalidad de los registros presentes en el dataset.

---

### Resultados

| Métrica | Valor |
|----------|---------|
| Registros Totales | 10,001,358 |
| Registros Duplicados | 0 |
| Porcentaje de Duplicados | 0% |

---

### Hallazgos

No se identificaron registros completamente duplicados dentro del dataset.

Este resultado confirma la consistencia estructural de la información almacenada y garantiza que cada observación mensual representa una entidad única dentro del historial de los créditos.

---

### Interpretación de Negocio

La ausencia de duplicados resulta especialmente relevante debido a la naturaleza histórica y periódica del dataset.

Cada registro representa el estado mensual de un crédito y su correcta identificación es fundamental para el análisis de evolución temporal y comportamiento financiero.

---

### Conclusión

El dataset presenta una estructura consistente y libre de registros duplicados, fortaleciendo su confiabilidad como fuente principal para el seguimiento histórico de préstamos POS y Cash Loans.

# 9. Variables de Negocio Relevantes

Se analizan las variables con mayor relevancia para el dominio financiero.

In [14]:
important_cols = [
    "SK_ID_PREV",
    "SK_ID_CURR",
    "MONTHS_BALANCE",
    "CNT_INSTALMENT",
    "CNT_INSTALMENT_FUTURE",
    "NAME_CONTRACT_STATUS",
    "SK_DPD"
]

df[important_cols].head()

NameError: name 'df' is not defined

In [15]:
df["NAME_CONTRACT_STATUS"].value_counts(normalize=True).mul(100).round(2)

NameError: name 'df' is not defined

## Interpretación

Las variables seleccionadas representan los principales indicadores asociados al seguimiento y evolución de préstamos POS y Cash Loans dentro del ecosistema Home Credit.

A diferencia de otros datasets enfocados en solicitudes o pagos específicos, POS_CASH_balance permite observar el estado mensual de los créditos y su avance dentro del plan de pagos.

---

### Variables Identificadas

| Variable | Descripción |
|-----------|-------------|
| SK_ID_PREV | Identificador del crédito asociado |
| SK_ID_CURR | Identificador del cliente |
| MONTHS_BALANCE | Mes relativo de observación |
| CNT_INSTALMENT | Número total de cuotas del crédito |
| CNT_INSTALMENT_FUTURE | Número de cuotas pendientes |
| NAME_CONTRACT_STATUS | Estado contractual del crédito |
| SK_DPD | Días de atraso |

---

### Relevancia de Negocio

#### SK_ID_PREV

Permite identificar el crédito asociado a cada observación mensual.

Será utilizado para reconstruir la evolución histórica de los préstamos.

---

#### SK_ID_CURR

Permite relacionar el comportamiento observado con el cliente correspondiente.

Esta variable será utilizada posteriormente para generar indicadores agregados de comportamiento financiero.

---

#### MONTHS_BALANCE

Representa el período temporal de observación.

Permite analizar la evolución histórica de los créditos y su comportamiento a lo largo del tiempo.

---

#### CNT_INSTALMENT

Corresponde al número total de cuotas definidas para el crédito.

Esta variable permite caracterizar la duración original del préstamo.

---

#### CNT_INSTALMENT_FUTURE

Representa el número de cuotas pendientes en cada período observado.

Su evolución permite medir el avance del crédito y el cumplimiento del plan de pagos.

---

#### NAME_CONTRACT_STATUS

Describe la situación operativa del crédito durante cada observación.

Permite segmentar los préstamos según su estado contractual.

---

#### SK_DPD

Representa los días de atraso registrados durante el período.

Esta variable constituye una señal directa de incumplimiento financiero y riesgo crediticio.

---

### Hallazgos Iniciales

La muestra evidencia que los créditos pueden encontrarse en diferentes etapas de su ciclo de vida.

Asimismo, la diferencia entre:

- CNT_INSTALMENT
- CNT_INSTALMENT_FUTURE

permite identificar el progreso alcanzado dentro del plan de pagos.

---

### Indicadores Potenciales

A partir de estas variables será posible construir métricas relacionadas con:

- Avance del crédito.
- Cuotas pendientes promedio.
- Historial de mora.
- Frecuencia de atraso.
- Cumplimiento financiero.
- Evolución de créditos activos.

---

### Conclusión

Las variables analizadas constituyen la base para comprender la evolución mensual de los préstamos POS y Cash Loans.

Su combinación permitirá generar indicadores relevantes para la evaluación del comportamiento financiero y riesgo crediticio dentro de las futuras capas analíticas de la arquitectura Lakehouse.

# 10. Variables Categóricas

In [16]:
categorical_columns = df.select_dtypes(include=["object"])

categorical_columns.columns.tolist()

NameError: name 'df' is not defined

In [17]:
for col in categorical_columns.columns:
    print(f"{col}: {df[col].nunique()}")

NameError: name 'categorical_columns' is not defined

## Interpretación

Se realizó la identificación de variables categóricas presentes en el dataset con el objetivo de comprender los estados operativos asociados a los préstamos POS y Cash Loans.

---

### Variables Categóricas Identificadas

| Variable | Categorías |
|-----------|-----------|
| NAME_CONTRACT_STATUS | 9 |

---

### Distribución de Estados del Contrato

| Estado | Participación |
|----------|-------------|
| Active | 91.50% |
| Completed | 7.45% |
| Signed | 0.87% |
| Demand | 0.07% |
| Returned to the store | 0.05% |
| Approved | 0.05% |
| Amortized debt | 0.01% |
| Canceled | 0.00% |
| XNA | 0.00% |

---

### Interpretación General

La distribución observada evidencia que la gran mayoría de los registros corresponden a créditos activos.

Este comportamiento resulta consistente con la naturaleza histórica del dataset, ya que cada observación representa una fotografía mensual de créditos que permanecían vigentes durante el período analizado.

---

### Hallazgos Relevantes

#### Créditos Activos

Los registros con estado **Active** representan el 91.50% del total.

Esto indica que la mayor parte de las observaciones corresponden a préstamos que continuaban en ejecución.

---

#### Créditos Completados

Los créditos con estado **Completed** representan el 7.45% del total.

Estos registros corresponden a préstamos que finalizaron exitosamente su ciclo de vida.

---

#### Estados Minoritarios

Los estados:

- Signed
- Demand
- Returned to the store
- Approved
- Amortized debt
- Canceled
- XNA

presentan una participación marginal dentro del dataset.

No obstante, estos estados pueden aportar información relevante para procesos de segmentación y análisis de riesgo.

---

### Valor Analítico

La variable NAME_CONTRACT_STATUS permitirá:

- Segmentar créditos según su estado operativo.
- Analizar diferencias de comportamiento entre créditos activos y finalizados.
- Construir métricas de cumplimiento financiero.
- Evaluar patrones asociados al ciclo de vida de los préstamos.

---

### Conclusión

Aunque el dataset incorpora una única variable categórica, su relevancia analítica es significativa debido a que describe el estado operativo de los créditos observados durante cada período.

## Interpretación

El dataset POS_CASH_balance almacena el comportamiento histórico mensual asociado a préstamos POS y Cash Loans otorgados a los clientes de Home Credit.

Su función principal es proporcionar visibilidad sobre la evolución financiera y operativa de los créditos a lo largo del tiempo.

---

### Relación Principal

El dataset se relaciona con las demás fuentes mediante dos identificadores clave:

| Variable | Relación |
|-----------|----------|
| SK_ID_CURR | Cliente |
| SK_ID_PREV | Crédito |

---

### Relación con application_train

application_train (1)

↓

POS_CASH_balance (N)

Un cliente puede registrar múltiples observaciones históricas asociadas a distintos créditos.

---

### Relación con previous_application

previous_application (1)

↓

POS_CASH_balance (N)

Un crédito puede generar múltiples observaciones mensuales durante su ciclo de vida.

---

### Jerarquía de Información

Cliente

↓

Solicitud Actual

↓

Crédito Histórico

↓

Seguimiento Mensual del Crédito

---

### Granularidad

1 registro = 1 crédito en 1 período mensual

La combinación:

- SK_ID_PREV
- MONTHS_BALANCE

identifica de forma única cada observación dentro del dataset.

---

### Integración con el Modelo Dimensional

El dataset constituye una fuente importante para la construcción de indicadores relacionados con:

- Evolución de créditos.
- Cumplimiento de cuotas.
- Historial de mora.
- Avance del plan de pagos.
- Riesgo financiero.

---

### Conclusión

POS_CASH_balance complementa la visión histórica de los créditos y permite monitorear su comportamiento a lo largo del tiempo mediante observaciones mensuales.

# 12. Conclusiones Técnicas

## Resumen Ejecutivo

El dataset POS_CASH_balance almacena el historial mensual asociado a préstamos POS y Cash Loans otorgados por Home Credit.

Su objetivo principal es registrar la evolución operativa de los créditos, permitiendo analizar el avance de las cuotas, el estado contractual y los niveles de atraso observados durante su ciclo de vida.

---

## Métricas Generales

| Métrica | Valor |
|----------|---------|
| Registros | 10,001,358 |
| Columnas | 8 |
| Variables Numéricas | 7 |
| Variables Categóricas | 1 |
| Duplicados | 0 |

---

## Calidad de Datos

El dataset presenta un nivel de calidad muy elevado.

Las validaciones realizadas evidenciaron:

- Ausencia de registros duplicados.
- Ausencia de columnas con niveles críticos de valores faltantes.
- Consistencia estructural entre variables relacionadas con cuotas y seguimiento de créditos.

Los únicos valores faltantes identificados corresponden a:

- CNT_INSTALMENT
- CNT_INSTALMENT_FUTURE

ambos con una incidencia inferior al 0.3%.

---

## Granularidad

Cada registro representa:

**1 crédito en 1 período mensual específico**

La combinación:

- SK_ID_PREV
- MONTHS_BALANCE

garantiza la unicidad de las observaciones dentro del dataset.

---

## Relaciones Identificadas

### Relación con application_train

application_train (1)

↓

POS_CASH_balance (N)

---

### Relación con previous_application

previous_application (1)

↓

POS_CASH_balance (N)

Estas relaciones permiten reconstruir la evolución histórica de los créditos asociados a cada cliente.

---

## Variables de Negocio Relevantes

Las variables de mayor valor analítico identificadas fueron:

- MONTHS_BALANCE
- CNT_INSTALMENT
- CNT_INSTALMENT_FUTURE
- NAME_CONTRACT_STATUS
- SK_DPD
- SK_DPD_DEF

Estas variables permiten caracterizar el avance del crédito, el cumplimiento financiero y el riesgo asociado.

---

## Hallazgos Principales

### Estado de los Créditos

| Estado | Participación |
|----------|-------------|
| Active | 91.50% |
| Completed | 7.45% |
| Otros estados | < 1% |

La mayoría de los registros corresponden a créditos activos, confirmando que el dataset se encuentra orientado principalmente al seguimiento operativo de préstamos vigentes.

---

### Estructura Histórica

El dataset registra observaciones mensuales de los créditos, permitiendo analizar su evolución temporal y el progreso alcanzado dentro del plan de pagos.

---

### Valor Analítico

La información contenida en POS_CASH_balance permite analizar:

- Avance del crédito.
- Cuotas pendientes.
- Historial de mora.
- Frecuencia de atraso.
- Evolución contractual.
- Cumplimiento financiero.

---

## Rol dentro del Modelo Dimensional

El dataset es candidato para enriquecer indicadores relacionados con comportamiento histórico de créditos.

### FACT_POS_CASH_BALANCE

A partir de esta información podrán construirse métricas como:

- Cuotas pendientes promedio.
- Avance promedio de créditos.
- Frecuencia de mora.
- Historial de atraso.
- Créditos activos por cliente.

---

## Rol dentro de la Arquitectura Lakehouse

POS_CASH_balance.csv

↓

Bronze (ingesta cruda)

↓

Silver (validación y normalización)

↓

Intermediate (agregaciones históricas)

↓

Gold (indicadores de comportamiento financiero)

↓

Diamond (convergencia corporativa)

---

## Recomendaciones para la Siguiente Fase

1. Integrar POS_CASH_balance con previous_application mediante SK_ID_PREV.
2. Consolidar métricas históricas por cliente utilizando SK_ID_CURR.
3. Construir indicadores asociados al avance de créditos y cuotas pendientes.
4. Incorporar métricas de mora y cumplimiento financiero dentro de las capas Gold y Diamond.
5. Diseñar agregaciones temporales para el análisis histórico de préstamos.

---

## Conclusión Final

POS_CASH_balance constituye una fuente confiable para analizar la evolución mensual de préstamos POS y Cash Loans.

Su estructura histórica, calidad de datos y simplicidad operativa permiten construir indicadores robustos de cumplimiento financiero, avance de créditos y riesgo asociado al comportamiento de los clientes.